## Setup
[Link to Question](https://leetcode.com/problems/immediate-food-delivery-ii/?envType=study-plan-v2&envId=top-sql-50)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from datetime import date

# Initialize Spark Session
spark = SparkSession.builder.appName("LeetCode1174").getOrCreate()

# Define Schema
schema = StructType([
    StructField("delivery_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("customer_pref_delivery_date", DateType(), False)
])

# Example Data from LeetCode
data = [
    (1, 1, date(2019, 8, 1), date(2019, 8, 2)),
    (2, 2, date(2019, 8, 2), date(2019, 8, 2)),
    (3, 1, date(2019, 8, 11), date(2019, 8, 12)),
    (4, 3, date(2019, 8, 24), date(2019, 8, 24)),
    (5, 3, date(2019, 8, 21), date(2019, 8, 22)),
    (6, 2, date(2019, 8, 11), date(2019, 8, 13)),
    (7, 4, date(2019, 8, 9), date(2019, 8, 9))
]

# Create DataFrame
delivery_df = spark.createDataFrame(data, schema)

# Show input
delivery_df.show()

+-----------+-----------+----------+---------------------------+
|delivery_id|customer_id|order_date|customer_pref_delivery_date|
+-----------+-----------+----------+---------------------------+
|          1|          1|2019-08-01|                 2019-08-02|
|          2|          2|2019-08-02|                 2019-08-02|
|          3|          1|2019-08-11|                 2019-08-12|
|          4|          3|2019-08-24|                 2019-08-24|
|          5|          3|2019-08-21|                 2019-08-22|
|          6|          2|2019-08-11|                 2019-08-13|
|          7|          4|2019-08-09|                 2019-08-09|
+-----------+-----------+----------+---------------------------+



## Code

In [9]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

windowSpec = Window.partitionBy(F.col('customer_id')).orderBy('order_date')

ranked_df = delivery_df.withColumn('rn', F.rank().over(windowSpec)).filter(F.col('rn') == 1)

immediate_percentage = ranked_df.select(\
                                F.round(\
                                F.sum(F.when(F.col('order_date') == F.col('customer_pref_delivery_date'), 1)\
                                .otherwise(0))* 100.0 \
                                / F.count('delivery_id'),2).alias('immediate_percentage '))
immediate_percentage.show()

+---------------------+
|immediate_percentage |
+---------------------+
|                 50.0|
+---------------------+

